# Skill-Centric Grounded Reasoning — single instance

This notebook runs **one deterministic end-to-end pass** through the pipeline from the executive summary:

Question → classify → **bulletin-retriever** → **plaintext-table-parser** → **arithmetic-verifier** → provenance bundle.

`cross-doc-aggregator` is included as a no-op when only one bulletin is needed.

**Corpus:** repo `corpus/` mirrors competition layout (`index.txt` + `treasury_bulletin_YYYY_MM.txt`). In Docker that maps to `/app/corpus/`.

**Skills:** contracts live under `skills/*/SKILL.md` (Hermes-style); this notebook implements the same stages in plain Python so you can inspect intermediate JSON.

## 0. Paths and imports (stdlib only)

In [ ]:
from __future__ import annotations

import json
import re
import subprocess
from dataclasses import dataclass, field
from decimal import Decimal
from pathlib import Path

# Repo root = parent of notebooks/
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

CORPUS_ROOT = REPO_ROOT / "corpus"
SKILLS_ROOT = REPO_ROOT / "skills"
INDEX_PATH = CORPUS_ROOT / "index.txt"

assert INDEX_PATH.is_file(), f"Missing {INDEX_PATH} — run from repo root or set REPO_ROOT"
print("CORPUS_ROOT:", CORPUS_ROOT.resolve())

## 1. Instance: question + expected grounding shape

We use a toy arithmetic question that requires **two evidence rows**, explicit **units** from the caption, and a **calculation trace**.

In [ ]:
QUESTION = (
    "In the January 1953 Treasury bulletin excerpt, what is the sum of the Outstanding column "
    "(in millions of dollars) for categories Alpha and Beta?"
)

print(QUESTION)

## 2. Classify intent / date / topic (deterministic heuristics)

Production systems can use richer rules or a frozen classifier; here we extract **year**, **month**, and **topic tokens** for retrieval.

In [ ]:
MONTHS = {
    "january": 1,
    "february": 2,
    "march": 3,
    "april": 4,
    "may": 5,
    "june": 6,
    "july": 7,
    "august": 8,
    "september": 9,
    "october": 10,
    "november": 11,
    "december": 12,
}


def classify(q: str) -> dict:
    ql = q.lower()
    year_m = re.search(r"\b(19\d{2}|20\d{2})\b", q)
    year = int(year_m.group(1)) if year_m else None
    month = None
    for name, num in MONTHS.items():
        if name in ql:
            month = num
            break
    topics = []
    for t in ("treasury", "bulletin", "outstanding", "alpha", "beta"):
        if t in ql:
            topics.append(t)
    return {
        "year": year,
        "month": month,
        "topics": topics,
        "needs_arithmetic": any(w in ql for w in ("sum", "total", "percent", "growth", "change")),
    }


classification = classify(QUESTION)
print(json.dumps(classification, indent=2))

## 3. Skill: `bulletin-retriever`

Map normalized period → candidate files using `index.txt`, then optionally **`grep`** inside files (subprocess) to match topic cues — same idea as `/app/corpus` + shell tools in-container.

In [ ]:
def read_index(index_path: Path) -> list[str]:
    lines = []
    for line in index_path.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        lines.append(s)
    return lines


def match_bulletin_name(year: int | None, month: int | None, filename: str) -> bool:
    m = re.match(r"treasury_bulletin_(\d{4})_(\d{2})\.txt$", filename)
    if not m:
        return False
    fy, fm = int(m.group(1)), int(m.group(2))
    if year is not None and fy != year:
        return False
    if month is not None and fm != month:
        return False
    return True


def grep_hits(path: Path, pattern: str) -> list[tuple[int, str]]:
    try:
        proc = subprocess.run(
            ["grep", "-n", "-E", pattern, str(path)],
            capture_output=True,
            text=True,
            check=False,
        )
    except FileNotFoundError:
        hits = []
        rx = re.compile(pattern)
        for i, line in enumerate(path.read_text(encoding="utf-8").splitlines(), start=1):
            if rx.search(line):
                hits.append((i, line))
        return hits
    hits = []
    for line in proc.stdout.splitlines():
        num, rest = line.split(":", 1)
        hits.append((int(num), rest))
    return hits


def bulletin_retrieve(q: str, classification: dict, corpus: Path) -> dict:
    entries = read_index(corpus / "index.txt")
    year, month = classification.get("year"), classification.get("month")
    cands = [e for e in entries if match_bulletin_name(year, month, Path(e).name)]
    rationale = "Filtered index.txt by treasury_bulletin_YYYY_MM pattern"
    confidence = "high" if cands else "low"
    if not cands:
        cands = entries
        rationale += "; fallback to full index (no date filter hit)"
    ranked = []
    for rel in cands:
        p = corpus / rel
        if not p.is_file():
            continue
        overlap = sum(1 for t in classification.get("topics", []) if grep_hits(p, re.escape(t)))
        ranked.append((overlap, rel, p))
    ranked.sort(key=lambda x: (-x[0], x[1]))
    ordered = [{"path": r[1], "rationale": rationale, "confidence": confidence, "lex_hits": r[0]} for r in ranked]
    return {"candidates": ordered, "primary_path": ordered[0]["path"] if ordered else None}


retrieval = bulletin_retrieve(QUESTION, classification, CORPUS_ROOT)
print(json.dumps(retrieval, indent=2))

## 4. Skill: `cross-doc-aggregator`

For this instance we only need **one** bulletin; the aggregator returns that file unchanged and records that no merge was required.

In [ ]:
def cross_doc_aggregate(retrieval: dict) -> dict:
    paths = [c["path"] for c in retrieval.get("candidates", [])]
    primary = retrieval.get("primary_path")
    return {
        "mode": "single_doc" if len(paths) <= 1 else "multi_doc",
        "sources": paths[:1] or paths,
        "primary_path": primary,
    }


aggregation = cross_doc_aggregate(retrieval)
print(json.dumps(aggregation, indent=2))

## 5. Skill: `plaintext-table-parser`

Locate the demo table, parse **Outstanding** with **line numbers**, normalize **parentheses = negative**, and skip the printed **Total** row so we do not double-count.

In [ ]:
ROW_RE = re.compile(
    r"^\s{2}(?P<label>[A-Za-z][A-Za-z0-9 /\-]{0,60}?)\s{2,}(?P<raw>[\d,()]+)\s*$"
)


def parse_number_token(raw: str) -> Decimal:
    s = raw.strip()
    neg = s.startswith("(") and s.endswith(")")
    if neg:
        s = s[1:-1]
    s = s.replace(",", "")
    v = Decimal(s)
    return -v if neg else v


def parse_plaintext_table(path: Path) -> dict:
    lines = path.read_text(encoding="utf-8").splitlines()
    unit = None
    table_start = None
    for i, line in enumerate(lines):
        if "millions of dollars" in line.lower():
            unit = "millions_of_dollars"
        if line.strip().startswith("Table 4"):
            table_start = i
            break
    rows = []
    if table_start is None:
        return {"unit": unit, "rows": [], "error": "table anchor not found"}
    for j in range(table_start, len(lines)):
        line = lines[j]
        if line.strip().startswith("Notes:"):
            break
        m = ROW_RE.match(line)
        if not m:
            continue
        label = m.group("label").strip()
        low = label.lower()
        if low.startswith("total"):
            continue
        rows.append(
            {
                "line": j + 1,
                "label": label,
                "outstanding": str(parse_number_token(m.group("raw"))),
                "raw": m.group("raw"),
            }
        )
    return {
        "source_file": str(path.name),
        "unit": unit or "unknown",
        "table_caption_line": table_start + 1,
        "rows": rows,
    }


primary = CORPUS_ROOT / aggregation["primary_path"]
parsed = parse_plaintext_table(primary)
print(json.dumps(parsed, indent=2))

## 6. Skill: `arithmetic-verifier`

Deterministic **Decimal** sum for Alpha + Beta with an explicit trace (no LLM math).

In [ ]:
def arithmetic_sum_labels(parsed: dict, labels: set[str]) -> dict:
    unit = parsed.get("unit")
    selected = [r for r in parsed["rows"] if r["label"] in labels]
    if len(selected) != len(labels):
        return {"error": "missing labels", "wanted": sorted(labels), "have": [r["label"] for r in parsed["rows"]]}
    trace = []
    acc = Decimal("0")
    for r in selected:
        v = Decimal(r["outstanding"])
        trace.append({"op": "add", "arg_line": r["line"], "arg_label": r["label"], "value": str(v)})
        acc += v
    trace.append({"op": "total", "result": str(acc)})
    return {"value": str(acc), "unit": unit, "trace": trace}


arith = arithmetic_sum_labels(parsed, {"Alpha", "Beta"})
print(json.dumps(arith, indent=2))

## 7. Provenance bundle (grounding artifact)

Everything a verifier or human needs: **source file**, **line numbers**, **units**, **calculation trace**.

In [ ]:
def build_answer_payload(question: str, classification, retrieval, aggregation, parsed, arith) -> dict:
    evidence = [
        {
            "file": parsed["source_file"],
            "lines": [r["line"] for r in parsed["rows"] if r["label"] in ("Alpha", "Beta")],
            "rows": [r for r in parsed["rows"] if r["label"] in ("Alpha", "Beta")],
        }
    ]
    return {
        "question": question,
        "classification": classification,
        "retrieval": retrieval,
        "aggregation": aggregation,
        "parsed_table": parsed,
        "arithmetic": arith,
        "answer": {
            "value": arith.get("value"),
            "unit": arith.get("unit"),
            "narrative": (
                f"Sum of Outstanding for Alpha and Beta = {arith.get('value')} "
                f"({arith.get('unit', 'unknown')})."
            ),
        },
        "provenance": {
            "source_files": [parsed["source_file"]],
            "line_spans": evidence[0]["lines"],
            "units": parsed.get("unit"),
            "calculation_trace": arith.get("trace"),
        },
    }


payload = build_answer_payload(QUESTION, classification, retrieval, aggregation, parsed, arith)
print(json.dumps(payload, indent=2))

## 8. Optional: load `SKILL.md` text into the notebook

Use this to keep human reasoning aligned with the frozen agent contract.

In [ ]:
def load_skill(name: str) -> str:
    p = SKILLS_ROOT / name / "SKILL.md"
    return p.read_text(encoding="utf-8")


print(load_skill("arithmetic-verifier")[:800] + "\n...")

## 9. One-shot runner (full pipeline)

Re-run from a single cell when tweaking heuristics.

In [ ]:
def run_pipeline(question: str, corpus: Path) -> dict:
    c = classify(question)
    r = bulletin_retrieve(question, c, corpus)
    a = cross_doc_aggregate(r)
    path = corpus / a["primary_path"]
    p = parse_plaintext_table(path)
    # Demo question always sums Alpha+Beta; production would route on intent.
    ar = arithmetic_sum_labels(p, {"Alpha", "Beta"})
    return build_answer_payload(question, c, r, a, p, ar)


full = run_pipeline(QUESTION, CORPUS_ROOT)
assert full["answer"]["value"] == "800"
assert full["provenance"]["units"] == "millions_of_dollars"
print("OK — pipeline instance complete.")
full["answer"]